In [2]:
from ms_var_prediction import (
    OUTPUT_DIR,
    GaussianMixtureVaR,
    MarkovSwitchingVaR,
    batch_backtest_model,
    fetch_valid_sp500_tickers,
    settings,
)
from typing import List
from yfinance import Ticker

In this notebook we will run a Value-at-Risk backtest for `GaussianMixtureModel` from `sklearn` and our implentation of `MarkovSwitchingModel` and compare the results.

# Implementation

In [3]:
def run_gmm(tickers: List[Ticker]) -> None:
    gmm_model = GaussianMixtureVaR(max_iter=1000)
    gmm_res = batch_backtest_model(
        gmm_model,
        tickers,
        settings.START_DATE,
        settings.END_DATE,
        settings.ALPHA,
        settings.WINDOW_SHAPE,
    )
    gmm_res.to_csv(
        OUTPUT_DIR / f"backtesting_gmm_{settings.START_DATE}_{settings.END_DATE}"
    )
    return gmm_res

def run_msm(tickers: List[Ticker]) -> None:
    msm_model = MarkovSwitchingVaR(optimizer_options={"max_iter": 1000})
    msm_res = batch_backtest_model(
        msm_model,
        tickers,
        settings.START_DATE,
        settings.END_DATE,
        settings.ALPHA,
        settings.WINDOW_SHAPE,
    )
    msm_res.to_csv(
        OUTPUT_DIR / f"backtesting_msm_{settings.START_DATE}_{settings.END_DATE}"
    )
    return msm_res


# Backtest run

Let's download the data for the 2010-2025 years and run our models on this period. We will use only the tickers which have the full history on the selected time range.

In [4]:
def run_backtest(start_date: str, end_date: str):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tickers = fetch_valid_sp500_tickers(start_date, end_date)
    gmm_res_df = run_gmm(tickers)
    msm_res_df = run_msm(tickers)

    return gmm_res_df, msm_res_df

gmm_res_df, msm_res_df = run_backtest(settings.START_DATE, settings.END_DATE)

Yahoo Finance Tickers Backtested:   5%|▍         | 24/498 [19:27<4:48:31, 36.52s/it]

Error processing AMCR: division by zero


Yahoo Finance Tickers Backtested:   7%|▋         | 36/498 [29:19<4:56:35, 38.52s/it]

Error processing AON: division by zero


Yahoo Finance Tickers Backtested:  23%|██▎       | 114/498 [1:34:33<3:02:45, 28.55s/it]

...

Yahoo Finance Tickers Backtested:  53%|█████▎    | 264/498 [3:21:26<2:29:11, 38.26s/it]

Error processing K: division by zero


Yahoo Finance Tickers Backtested:  64%|██████▍   | 321/498 [3:55:49<1:18:45, 26.70s/it]

Error processing MNST: division by zero


Yahoo Finance Tickers Backtested: 100%|██████████| 498/498 [5:45:41<00:00, 41.65s/it]  


Now we can check the result.

In [5]:
print(f"Results for GMM: {gmm_res_df.sum().to_dict()}")
print(f"Results for MSM: {msm_res_df.sum().to_dict()}")

Results for GMM: {'binomial': 484, 'independence_simple': 67, 'kupiec': 484, 'christoffersen_independence': 67, 'christoffersen_conditional': 89}
Results for MSM: {'binomial': 430, 'independence_simple': 391, 'kupiec': 430, 'christoffersen_independence': 387, 'christoffersen_conditional': 369}


**Conclusion.** We can see that `MarkovSwitchingModel` performs significantly better with independence tests due to the ability to catch the current regime. 